# Cloudmake Kaggle job

This private notebook runs exactly one project-provided Make target.

In [ ]:
from pathlib import Path
import base64
import json
import subprocess
import sys

root = Path('/tmp/cloudmake-control')
root.mkdir(parents=True, exist_ok=True)
control = root / 'control.json'
source_archive = root / 'source.tar.gz'
remote_helper = root / 'kaggle_remote.py'
oci_helper = root / 'oci_runner.py'
control.write_bytes(base64.b64decode('__KAGGLE_CONTROL_B64__'))
source_archive.write_bytes(base64.b64decode('__SOURCE_ARCHIVE_B64__'))
remote_helper.write_bytes(base64.b64decode('__KAGGLE_REMOTE_HELPER_B64__'))
oci_helper.write_bytes(base64.b64decode('__OCI_RUNNER_HELPER_B64__'))


## Restore and execute

A selected workspace is restored from the preceding private kernel output. The helper overlays the current local source snapshot and executes the requested target once.

In [ ]:
completed = subprocess.run([
    sys.executable, str(remote_helper),
    '--control', str(control),
    '--source-archive', str(source_archive),
    '--oci-runner', str(oci_helper),
])


## Validate the execution boundary

Expected project target failures are recorded for the host without a notebook traceback. Missing or infrastructure-failed receipts remain notebook errors.

In [ ]:
receipt_path = Path('/kaggle/working/cloudmake-target-result.json')
try:
    receipt = json.loads(receipt_path.read_text())
except Exception as error:
    raise RuntimeError(f'Cloudmake runner produced no valid result receipt: {error}')
if completed.returncode or receipt.get('status') == 'infrastructure-failed':
    raise RuntimeError(receipt.get('error', f'Cloudmake runner exited with status {completed.returncode}'))
